# Agentic Finance — Agentic RAG over the Distributional Shield

This notebook adds an **agentic RAG layer** to the distributional-shield framework.

The design goal is simple:

\[
\text{retrieve evidence} \rightarrow \text{agent review} \rightarrow \text{shield-aware answer} \rightarrow \text{audit log}.
\]

Trade-specific answers are **composed from the retrieved evidence rows** — the agent quotes the highest-ranked shield decision for the symbol (and size) in question, and escalates when that evidence is weak, missing, or contradictory. Conceptual questions are answered with fixed templates that cite the retrieved context. The RAG agent can explain, cite, summarize and escalate. It **cannot weaken** deterministic or distributional controls.

In [1]:
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path.cwd()
# Resolve the shield evidence logs: next to this notebook, or in the repository's data/ folder.
DATA_DIR = BASE_DIR if (BASE_DIR / "agentic_finance_giant_notebook_audit_log.csv").exists() else BASE_DIR.parent / "data"
print(f"Evidence logs: {'alongside the notebook' if DATA_DIR == BASE_DIR else 'repository data/ folder'}")

Evidence logs: repository data/ folder


## 1. Load the shield evidence logs

The RAG corpus is grounded in the CSV outputs of the canonical distributional-shield notebook.

In [2]:
audit = pd.read_csv(DATA_DIR / "agentic_finance_giant_notebook_audit_log.csv")
committee = pd.read_csv(DATA_DIR / "agentic_finance_multiagent_decision_log.csv")
opinions = pd.read_csv(DATA_DIR / "agentic_finance_multiagent_opinion_log.csv")
outcomes = pd.read_csv(DATA_DIR / "agentic_finance_agent_negotiator_shield_outcomes.csv")
offers = pd.read_csv(DATA_DIR / "agentic_finance_agent_negotiator_shield_offers.csv")

# Fail closed on degraded evidence: a present-but-empty log would otherwise
# silently produce a hollow corpus that retrieves nothing useful.
for name, frame in [("audit", audit), ("committee", committee), ("opinions", opinions), ("outcomes", outcomes), ("offers", offers)]:
    if frame.empty:
        raise ValueError(f"Evidence log '{name}' is empty; refusing to build a degraded corpus.")

print("Loaded source logs:")
print("audit", audit.shape)
print("committee", committee.shape)
print("opinions", opinions.shape)
print("outcomes", outcomes.shape)
print("offers", offers.shape)

Loaded source logs:
audit (38, 31)
committee (4, 18)
opinions (16, 19)
outcomes (6, 21)
offers (33, 31)


## 2. Build the RAG corpus

Each corpus row is a small, citeable evidence unit with its decision `status` carried alongside the text. The corpus mixes conceptual assumptions with row-level audit evidence, committee decisions, negotiated outcomes, per-round offers, and per-agent committee opinions.

Numeric fields are formatted defensively (missing values become `n/a`, never the string `nan`), and notionals are also written in humanized form (`800000 (800k)`) so size-specific questions retrieve the right rows.

In [3]:
def fmt(value, precision: int = 2) -> str:
    """NaN-safe numeric formatting: evidence text must never contain 'nan'."""
    if value is None or pd.isna(value):
        return "n/a"
    return f"{value:,.{precision}f}"


def fmt_notional(value) -> str:
    """Notional with a humanized alias so '800k'-style questions match."""
    if value is None or pd.isna(value):
        return "n/a"
    if value >= 1_000_000 and value % 1_000_000 == 0:
        return f"{value:,.0f} ({value / 1_000_000:.0f}m)"
    if value >= 1_000:
        return f"{value:,.0f} ({value / 1_000:.0f}k)"
    return f"{value:,.0f}"


def text_field(row, key) -> str:
    """NaN-safe string field access for sparse CSV columns."""
    value = row.get(key, "")
    return "" if value is None or pd.isna(value) else str(value)


manual_docs = [
    {
        "doc_id": "M001",
        "source_file": "inline_policy_docs",
        "source_type": "architecture",
        "symbol": "",
        "topic": "agentic_rag_architecture",
        "section": "core_pipeline",
        "status": "",
        "side": "",
        "notional_value": None,
        "text": "The architecture is LLM proposes, typed Pydantic schema validates semantic form, deterministic rules check institutional admissibility, distributional risk gates evaluate post-trade VaR, CVaR and tail-loss probability, and only then can execution routing proceed.",
    },
    {
        "doc_id": "M002",
        "source_file": "inline_policy_docs",
        "source_type": "governance",
        "symbol": "",
        "topic": "non_weakening_rule",
        "section": "multi_agent_invariant",
        "status": "",
        "side": "",
        "notional_value": None,
        "text": "The committee and negotiator can recommend, challenge, reduce, condition, escalate or reject a proposal, but they can never weaken the shield. Final status is the stricter of agent decision and distributional shield decision.",
    },
    {
        "doc_id": "M003",
        "source_file": "inline_policy_docs",
        "source_type": "risk_model",
        "symbol": "",
        "topic": "post_trade_distribution",
        "section": "risk_assumptions",
        "status": "",
        "side": "",
        "notional_value": None,
        "text": "Risk gates are defined on post-trade total portfolio risk. The implemented functionals are VaR 95, CVaR 95 and probability that loss exceeds the maximum loss threshold. These are the auditable governance controls together with deterministic rule names, reason strings, committee votes and negotiation offers.",
    },
    {
        "doc_id": "M004",
        "source_file": "inline_policy_docs",
        "source_type": "rag_policy",
        "symbol": "",
        "topic": "evidence_policy",
        "section": "rag_grounding",
        "status": "",
        "side": "",
        "notional_value": None,
        "text": "An agentic RAG answer should cite retrieved audit rows, committee decisions, negotiator outcomes or modeling assumptions. If evidence is weak, contradictory or missing, the answer must escalate rather than invent a trade recommendation.",
    },
]

rows = manual_docs.copy()

for i, row in audit.iterrows():
    rows.append({
        "doc_id": f"AUDIT_{i:03d}",
        "source_file": "agentic_finance_giant_notebook_audit_log.csv",
        "source_type": "single_shield_audit",
        "symbol": text_field(row, "symbol"),
        "topic": "distributional_shield_decision",
        "section": text_field(row, "scenario_set"),
        "status": text_field(row, "status"),
        "side": text_field(row, "side"),
        "notional_value": row.get("notional_value"),
        "text": (
            f"Audit scenario {text_field(row, 'scenario_set')} for {text_field(row, 'side')} {text_field(row, 'symbol')} "
            f"notional {fmt_notional(row.get('notional_value'))}: status {text_field(row, 'status')}. "
            f"Reason: {text_field(row, 'reason')} VaR_95 {fmt(row.get('VaR_95'))}, CVaR_95 {fmt(row.get('CVaR_95'))}, "
            f"tail probability {fmt(row.get('P_loss_gt_limit'), 4)}, max symbol weight {fmt(row.get('Max_symbol_weight_after'), 4)}, "
            f"gross exposure multiple {fmt(row.get('Gross_exposure_multiple_after'), 4)}. Rules checked: {text_field(row, 'rules')}."
        ),
    })

for i, row in committee.iterrows():
    rows.append({
        "doc_id": f"COMMITTEE_{i:03d}",
        "source_file": "agentic_finance_multiagent_decision_log.csv",
        "source_type": "committee_decision",
        "symbol": text_field(row, "symbol"),
        "topic": "multi_agent_governance",
        "section": "committee_final_status",
        "status": text_field(row, "final_status"),
        "side": text_field(row, "side"),
        "notional_value": row.get("notional_value"),
        "text": (
            f"Committee decision {text_field(row, 'committee_id')} for {text_field(row, 'side')} {text_field(row, 'symbol')} "
            f"notional {fmt_notional(row.get('notional_value'))}: consensus status {text_field(row, 'consensus_status')}, "
            f"final status {text_field(row, 'final_status')}. Consensus reason: {text_field(row, 'consensus_reason')} "
            f"Final reason: {text_field(row, 'final_reason')} Shield status {text_field(row, 'shield_status')} "
            f"because {text_field(row, 'shield_reason')}. Risk metrics: VaR_95 {fmt(row.get('VaR_95'))}, "
            f"CVaR_95 {fmt(row.get('CVaR_95'))}, tail probability {fmt(row.get('P_loss_gt_limit'), 4)}."
        ),
    })

for i, row in opinions.iterrows():
    rows.append({
        "doc_id": f"OPINION_{i:03d}",
        "source_file": "agentic_finance_multiagent_opinion_log.csv",
        "source_type": "committee_opinion",
        "symbol": text_field(row, "symbol"),
        "topic": "per_agent_committee_vote",
        "section": text_field(row, "role"),
        "status": text_field(row, "vote"),
        "side": text_field(row, "side"),
        "notional_value": row.get("notional_value"),
        "text": (
            f"Committee opinion for {text_field(row, 'side')} {text_field(row, 'symbol')} "
            f"notional {fmt_notional(row.get('notional_value'))}: agent {text_field(row, 'agent_name')} "
            f"role {text_field(row, 'role')} voted {text_field(row, 'vote')} with confidence {fmt(row.get('confidence'))}. "
            f"Reason: {text_field(row, 'reason')}"
        ),
    })

for i, row in outcomes.iterrows():
    rows.append({
        "doc_id": f"NEGOTIATOR_{i:03d}",
        "source_file": "agentic_finance_agent_negotiator_shield_outcomes.csv",
        "source_type": "negotiated_outcome",
        "symbol": text_field(row, "symbol"),
        "topic": "agent_negotiator_outcome",
        "section": "final_negotiated_terms",
        "status": text_field(row, "final_status"),
        "side": text_field(row, "side"),
        "notional_value": row.get("original_notional_value"),
        "text": (
            f"Negotiation session {text_field(row, 'session_id')} for {text_field(row, 'side')} {text_field(row, 'symbol')}: "
            f"original notional {fmt_notional(row.get('original_notional_value'))}, "
            f"negotiated notional {fmt_notional(row.get('negotiated_notional_value'))}, reduction {fmt(row.get('notional_reduction'))}, "
            f"negotiation status {text_field(row, 'negotiation_status')}, final status {text_field(row, 'final_status')}. "
            f"Negotiation reason: {text_field(row, 'negotiation_reason')} Final reason: {text_field(row, 'final_reason')} "
            f"Shield status {text_field(row, 'shield_status')} because {text_field(row, 'shield_reason')}. "
            f"Execution style {text_field(row, 'execution_style')}, child orders {fmt(row.get('child_order_count'), 0)}, "
            f"max participation rate {fmt(row.get('max_participation_rate'))}. "
            f"VaR_95 {fmt(row.get('VaR_95'))}, CVaR_95 {fmt(row.get('CVaR_95'))}, tail probability {fmt(row.get('P_loss_gt_limit'), 4)}."
        ),
    })

for i, row in offers.iterrows():
    rows.append({
        "doc_id": f"OFFER_{i:03d}",
        "source_file": "agentic_finance_agent_negotiator_shield_offers.csv",
        "source_type": "negotiation_offer",
        "symbol": text_field(row, "symbol"),
        "topic": "round_by_round_negotiation",
        "section": f"round_{int(row['round_index'])}",
        "status": text_field(row, "vote"),
        "side": text_field(row, "side"),
        "notional_value": row.get("original_notional_value"),
        "text": (
            f"Negotiation offer round {int(row['round_index'])} in session {text_field(row, 'session_id')} for {text_field(row, 'symbol')}: "
            f"agent {text_field(row, 'agent_name')} role {text_field(row, 'role')} voted {text_field(row, 'vote')} "
            f"with confidence {fmt(row.get('confidence'))}. Proposed notional {fmt_notional(row.get('proposed_notional_value'))}. "
            f"Reason: {text_field(row, 'reason')} "
            f"Execution style {text_field(row, 'term_execution_style')}; hard block {text_field(row, 'term_hard_block')}; "
            f"kill switch {text_field(row, 'term_kill_switch')}."
        ),
    })

corpus = pd.DataFrame(rows)
assert len(corpus) > 20, "Corpus is implausibly small; refusing to continue with degraded evidence."
corpus.to_csv(BASE_DIR / "agentic_rag_corpus.csv", index = False)
display(corpus.head(8))
print(corpus.shape)

,doc_id,source_file,source_type,symbol,topic,section,status,side,notional_value,text
0,M001,inline_policy_docs,architecture,,agentic_rag_architecture,core_pipeline,,,NaN,"The architecture is LLM proposes, typed Pydant..."
1,M002,inline_policy_docs,governance,,non_weakening_rule,multi_agent_invariant,,,NaN,"The committee and negotiator can recommend, ch..."
2,M003,inline_policy_docs,risk_model,,post_trade_distribution,risk_assumptions,,,NaN,Risk gates are defined on post-trade total por...
3,M004,inline_policy_docs,rag_policy,,evidence_policy,rag_grounding,,,NaN,An agentic RAG answer should cite retrieved au...
4,AUDIT_000,agentic_finance_giant_notebook_audit_log.csv,single_shield_audit,AAPL,distributional_shield_decision,core_examples,allow,BUY,"250,000.0000",Audit scenario core_examples for BUY AAPL noti...
5,AUDIT_001,agentic_finance_giant_notebook_audit_log.csv,single_shield_audit,TSLA,distributional_shield_decision,core_examples,reject,BUY,"800,000.0000",Audit scenario core_examples for BUY TSLA noti...
6,AUDIT_002,agentic_finance_giant_notebook_audit_log.csv,single_shield_audit,TSLA,distributional_shield_decision,core_examples,reject,BUY,"1,200,000.0000",Audit scenario core_examples for BUY TSLA noti...
7,AUDIT_003,agentic_finance_giant_notebook_audit_log.csv,single_shield_audit,GME,distributional_shield_decision,core_examples,reject,BUY,"50,000.0000",Audit scenario core_examples for BUY GME notio...


(101, 10)


## 3. Retrieval engine

For portability, the notebook uses local TF-IDF retrieval with optional **metadata filtering** (by symbol and source type). The filtered pass is what grounds trade-specific answers: lexical similarity alone can rank generically-worded documents above the decision rows that actually cover the named trade. The engine can be swapped for embeddings or a vector database without changing the agent contracts.

In [4]:
class RetrievalEngine:
    def __init__(self, corpus_df: pd.DataFrame):
        self.corpus = corpus_df.reset_index(drop = True)
        self.vectorizer = TfidfVectorizer(stop_words = "english", ngram_range = (1, 2), min_df = 1)
        self.matrix = self.vectorizer.fit_transform(self.corpus["text"].fillna(""))

    def retrieve(self, query: str, top_k: int = 5, symbols = None, source_types = None) -> pd.DataFrame:
        """Score all docs against the query; optionally pre-filter by metadata.

        Metadata filtering is the standard production-RAG answer to lexical misses:
        a question that names a symbol should be answered from that symbol's
        decision rows, not from whichever documents happen to share generic words.
        """
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.matrix).ravel()
        mask = np.ones(len(self.corpus), dtype = bool)
        if symbols is not None:
            mask &= self.corpus["symbol"].isin(list(symbols)).to_numpy()
        if source_types is not None:
            mask &= self.corpus["source_type"].isin(list(source_types)).to_numpy()
        candidates = np.flatnonzero(mask)
        order = candidates[np.argsort(scores[candidates])[::-1][:top_k]]
        result = self.corpus.iloc[order].copy()
        result["retrieval_score"] = scores[order]
        result["rank"] = np.arange(1, len(result) + 1)
        return result[["rank", "doc_id", "source_file", "source_type", "symbol", "side", "notional_value", "topic", "section", "status", "retrieval_score", "text"]]

engine = RetrievalEngine(corpus)
retrieved_demo = engine.retrieve("Can the agent execute an 800k TSLA buy under the distributional shield?", top_k = 5)
display(retrieved_demo[["rank", "doc_id", "source_type", "symbol", "retrieval_score", "topic"]])

,rank,doc_id,source_type,symbol,retrieval_score,topic
45,1,COMMITTEE_003,committee_decision,TLT,0.2263,multi_agent_governance
42,2,COMMITTEE_000,committee_decision,MSFT,0.2227,multi_agent_governance
1,3,M002,governance,,0.1728,non_weakening_rule
66,4,NEGOTIATOR_004,negotiated_outcome,TLT,0.1263,agent_negotiator_outcome
62,5,NEGOTIATOR_000,negotiated_outcome,MSFT,0.1229,agent_negotiator_outcome


## 4. Agentic review layer

The decision layer is **evidence-driven**. For a trade-specific question (one that names a symbol in the corpus), the agent runs a symbol-filtered retrieval pass over the decision-bearing rows — narrowed to the requested size when the question names one — and composes the answer from the highest-ranked row, quoting it as evidence. It fails closed three ways: evidence below the retrieval-score threshold escalates, evidence that does not cover the requested symbol or size escalates, and **contradictory** evidence (the same trade — symbol, side, and size — carrying both allow and reject verdicts) escalates. Conceptual questions use fixed answer templates that cite the retrieved context documents.

Four lightweight reviewer rows are derived from the same decision data for the audit trail:

1. `RetrieverAgent`: reports evidence strength (its confidence is the retrieval score).
2. `EvidenceQualityAgent`: reports grounding and source diversity (at least two source types required).
3. `DistributionalRiskAgent`: carries the evidence-derived status.
4. `ComplianceAgent`: forces fail-closed behavior and governance caveats.

In this teaching implementation the reviewers are deterministic functions of the evidence signals — in a production system they could be LLM agents — but the principle stands: the answer is grounded in retrieved rows, and weak grounding escalates rather than answers.

In [5]:
WEAK_SCORE_THRESHOLD = 0.12
DECISION_SOURCE_TYPES = {"single_shield_audit", "committee_decision", "negotiated_outcome"}


def extract_question_symbols(question: str) -> list[str]:
    """Symbols the question refers to, matched against the corpus symbol set."""
    q_upper = question.upper()
    known_symbols = sorted({s for s in corpus["symbol"].unique() if s})
    return [symbol for symbol in known_symbols if symbol in q_upper]


def extract_question_notional(question: str) -> str | None:
    """A humanized notional token like '800k' / '5m' if the question names a size."""
    match = re.search(r"(\d+(?:[.,]\d+)?)\s*(K|M)\b", question.upper())
    if not match:
        return None
    return f"{match.group(1).replace(',', '')}{match.group(2).lower()}"


def infer_decision(question: str, retrieved: pd.DataFrame) -> dict[str, Any]:
    q_upper = question.upper()
    evidence_ids = retrieved["doc_id"].tolist()
    top_score = float(retrieved["retrieval_score"].max()) if len(retrieved) else 0.0
    source_diversity = int(retrieved["source_type"].nunique()) if len(retrieved) else 0
    weak_evidence = top_score < WEAK_SCORE_THRESHOLD or source_diversity < 2

    question_symbols = extract_question_symbols(question)
    question_notional = extract_question_notional(question)

    base = {
        "weak_evidence": weak_evidence,
        "source_diversity": source_diversity,
        "top_score": top_score,
        "evidence_ids": " | ".join(evidence_ids),
    }

    # --- Trade-specific questions: the answer is composed from retrieved evidence. ---
    if question_symbols:
        if weak_evidence:
            return {**base, "final_status": "escalate", "risk_status": "weak_evidence",
                    "answer": "Evidence is too weak for a grounded answer about this trade. The agentic RAG policy requires escalation rather than invention."}

        # Evidence pass: a symbol-filtered retrieval over decision-bearing rows,
        # so the answer is grounded in the named trade's own shield decisions.
        evidence = engine.retrieve(
            question,
            top_k = 4,
            symbols = question_symbols,
            source_types = sorted(DECISION_SOURCE_TYPES),
        )
        if question_notional is not None:
            evidence = evidence[evidence["text"].str.contains(f"({question_notional})", regex = False)]
        base["evidence_ids"] = " | ".join(dict.fromkeys(evidence_ids + evidence["doc_id"].tolist()))

        if evidence.empty:
            return {**base, "final_status": "escalate", "risk_status": "evidence_gap",
                    "answer": (
                        f"The retrieved evidence does not cover the requested trade "
                        f"({', '.join(question_symbols)}{' at ' + question_notional if question_notional else ''}). "
                        "The RAG agent escalates rather than answering beyond its evidence."
                    )}

        # Contradiction means conflicting verdicts on the SAME trade (symbol, side,
        # size) — different sizes or instruments legitimately get different verdicts.
        statuses_by_trade = evidence.groupby(["symbol", "side", "notional_value"])["status"].agg(set)
        if any({"allow", "reject"} <= trade_statuses for trade_statuses in statuses_by_trade):
            return {**base, "final_status": "escalate", "risk_status": "contradictory_evidence",
                    "answer": (
                        "The retrieved evidence is contradictory: the same trade appears with both allow and reject "
                        "verdicts. The RAG agent escalates for human reconciliation rather than choosing a side."
                    )}

        top_evidence = evidence.iloc[0]
        citation = f"Evidence {top_evidence['doc_id']}: {top_evidence['text']}"
        if top_evidence["status"] == "reject":
            return {**base, "final_status": "reject", "risk_status": "distributional_reject",
                    "answer": f"No. The retrieved shield evidence rejects this trade, and the RAG agent returns the shield decision rather than overriding it. {citation}"}
        if top_evidence["status"] == "escalate":
            return {**base, "final_status": "escalate", "risk_status": "conditional_acceptance",
                    "answer": f"This trade requires human review according to the retrieved evidence. {citation}"}
        return {**base, "final_status": "answer", "risk_status": "shield_allowed",
                "answer": f"The retrieved evidence shows this trade was admitted by the shield. This answer is informational only; execution still requires the live shield. {citation}"}

    # --- Conceptual questions: fixed templates that cite the retrieved context. ---
    context = f" Context: {' | '.join(evidence_ids[:3])}."
    if "PYDANTIC" in q_upper or "TYPED" in q_upper:
        return {**base, "final_status": "answer", "risk_status": "conceptual",
                "answer": "Typed validation only proves semantic form: symbol, side and non-negative notional are well formed. Financial validity requires portfolio-aware deterministic and distributional checks on post-trade VaR, CVaR and tail-loss probability." + context}
    if "AUDIT" in q_upper or "GOVERNANCE" in q_upper or "CONTROLS" in q_upper:
        return {**base, "final_status": "answer", "risk_status": "governance",
                "answer": "Auditability comes from typed proposals, deterministic rule names, distributional metrics with their governing limits, final status, reason strings, committee votes, negotiation offers and preserved source-level evidence IDs." + context}
    if "INVENT" in q_upper or "WEAK" in q_upper:
        return {**base, "final_status": "escalate", "risk_status": "rag_fail_closed",
                "answer": "No. The RAG agent must fail closed: weak, missing or contradictory evidence leads to escalation, not invented trade advice." + context}
    if weak_evidence:
        return {**base, "final_status": "escalate", "risk_status": "weak_evidence",
                "answer": "Evidence is too weak for a grounded answer. The agentic RAG policy requires escalation rather than invention."}
    return {**base, "final_status": "answer", "risk_status": "informational",
            "answer": "The retrieved evidence supports an informational answer, but it does not authorize execution without the deterministic and distributional shield." + context}


def run_agentic_rag(question: str, query_id: str = "adhoc", top_k: int = 6) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    retrieved = engine.retrieve(question, top_k = top_k)
    decision = infer_decision(question, retrieved)

    retrieval_log = retrieved.copy()
    retrieval_log.insert(0, "query_id", query_id)

    # Reviewer rows are derived from the same evidence signals as the decision,
    # so the audit trail and the answer can never disagree.
    opinion_log = pd.DataFrame([
        {
            "query_id": query_id,
            "agent_name": "RetrieverAgent",
            "vote": "support" if decision["top_score"] >= WEAK_SCORE_THRESHOLD else "escalate",
            "confidence": round(min(0.99, decision["top_score"]), 4),
            "reason": f"Top retrieval score {decision['top_score']:.4f}; threshold {WEAK_SCORE_THRESHOLD}.",
        },
        {
            "query_id": query_id,
            "agent_name": "EvidenceQualityAgent",
            "vote": "support" if not decision["weak_evidence"] else "escalate",
            "confidence": 0.90 if not decision["weak_evidence"] else 0.55,
            "reason": f"Source diversity {decision['source_diversity']} (minimum 2); " + ("evidence is sufficiently grounded." if not decision["weak_evidence"] else "evidence is too weak; fail closed."),
        },
        {
            "query_id": query_id,
            "agent_name": "DistributionalRiskAgent",
            "vote": decision["final_status"],
            "confidence": 0.88,
            "reason": f"Risk status: {decision['risk_status']}",
        },
        {
            "query_id": query_id,
            "agent_name": "ComplianceAgent",
            "vote": "support" if decision["final_status"] != "reject" else "reject",
            "confidence": 0.86,
            "reason": "Do not bypass shield; include governance caveat.",
        },
    ])

    answer_log = pd.DataFrame([{
        "query_id": query_id,
        "user_question": question,
        **decision,
    }])

    return retrieval_log, opinion_log, answer_log

## 5. Example query set and generated CSV outputs

In [6]:
queries = pd.DataFrame([
    {"query_id": "Q001", "user_question": "Can the agent execute an 800k TSLA buy under the distributional shield?", "expected_focus": "TSLA hard risk rejection and tail probability"},
    {"query_id": "Q002", "user_question": "Explain why typed Pydantic validation is not enough for financial validity.", "expected_focus": "semantic validity versus distributional financial validity"},
    {"query_id": "Q003", "user_question": "What happens to the HY_CDS proposal after liquidity and execution negotiation?", "expected_focus": "HY_CDS liquidity cap staged VWAP child orders kill switch"},
    {"query_id": "Q004", "user_question": "What was the final negotiated NVDA outcome and why did it escalate?", "expected_focus": "NVDA negotiated notional CVaR escalation"},
    {"query_id": "Q005", "user_question": "Which controls make the framework auditable for governance?", "expected_focus": "audit rows rules checked source diversity final status"},
    {"query_id": "Q006", "user_question": "Should the RAG agent invent an answer when retrieved evidence is weak?", "expected_focus": "evidence policy fail closed escalation"},
])

queries.to_csv(BASE_DIR / "agentic_rag_queries.csv", index = False)

retrieval_logs = []
opinion_logs = []
answer_logs = []

for _, query_row in queries.iterrows():
    retrieval_log, opinion_log, answer_log = run_agentic_rag(
        question = query_row["user_question"],
        query_id = query_row["query_id"],
        top_k = 6,
    )
    retrieval_logs.append(retrieval_log)
    opinion_logs.append(opinion_log)
    answer_logs.append(answer_log)

retrieval_log_df = pd.concat(retrieval_logs, ignore_index = True)
opinion_log_df = pd.concat(opinion_logs, ignore_index = True)
answer_log_df = pd.concat(answer_logs, ignore_index = True)

retrieval_log_df.to_csv(BASE_DIR / "agentic_rag_retrieval_log.csv", index = False)
opinion_log_df.to_csv(BASE_DIR / "agentic_rag_agent_opinion_log.csv", index = False)
answer_log_df.to_csv(BASE_DIR / "agentic_rag_answer_log.csv", index = False)

display(answer_log_df[["query_id", "final_status", "risk_status", "top_score", "evidence_ids", "answer"]])

,query_id,final_status,risk_status,top_score,evidence_ids,answer
0,Q001,reject,distributional_reject,0.2263,COMMITTEE_003 | COMMITTEE_000 | M002 | NEGOTIA...,No. The retrieved shield evidence rejects this...
1,Q002,answer,conceptual,0.2564,M001 | AUDIT_032 | AUDIT_022 | AUDIT_023 | AUD...,Typed validation only proves semantic form: sy...
2,Q003,answer,shield_allowed,0.1957,OFFER_030 | OFFER_026 | NEGOTIATOR_003 | OFFER...,The retrieved evidence shows this trade was ad...
3,Q004,escalate,conditional_acceptance,0.2154,NEGOTIATOR_001 | NEGOTIATOR_002 | NEGOTIATOR_0...,This trade requires human review according to ...
4,Q005,answer,governance,0.2458,M003 | OFFER_032 | OFFER_024 | OFFER_015 | OFF...,"Auditability comes from typed proposals, deter..."
5,Q006,escalate,rag_fail_closed,0.4625,M004 | OPINION_007 | OPINION_003 | OPINION_015...,"No. The RAG agent must fail closed: weak, miss..."


## 6. Ad hoc question interface

Change `question` below to ask new governance, execution, or framework questions against the local corpus.

In [7]:
question = "Why does the framework say an agent can propose but not execute directly?"
retrieval_log, opinion_log, answer_log = run_agentic_rag(question = question, query_id = "ADHOC", top_k = 6)

display(retrieval_log[["rank", "doc_id", "source_type", "symbol", "retrieval_score", "topic"]])
display(opinion_log)
display(answer_log[["final_status", "risk_status", "evidence_ids", "answer"]])

,rank,doc_id,source_type,symbol,retrieval_score,topic
53,1,OPINION_007,committee_opinion,NVDA,0.0874,per_agent_committee_vote
49,2,OPINION_003,committee_opinion,MSFT,0.0863,per_agent_committee_vote
61,3,OPINION_015,committee_opinion,TLT,0.0862,per_agent_committee_vote
57,4,OPINION_011,committee_opinion,GME,0.0811,per_agent_committee_vote
52,5,OPINION_006,committee_opinion,NVDA,0.0793,per_agent_committee_vote
48,6,OPINION_002,committee_opinion,MSFT,0.0785,per_agent_committee_vote


,query_id,agent_name,vote,confidence,reason
0,ADHOC,RetrieverAgent,escalate,0.0874,Top retrieval score 0.0874; threshold 0.12.
1,ADHOC,EvidenceQualityAgent,escalate,0.5500,Source diversity 1 (minimum 2); evidence is to...
2,ADHOC,DistributionalRiskAgent,escalate,0.8800,Risk status: weak_evidence
3,ADHOC,ComplianceAgent,support,0.8600,Do not bypass shield; include governance caveat.


,final_status,risk_status,evidence_ids,answer
0,escalate,weak_evidence,OPINION_007 | OPINION_003 | OPINION_015 | OPIN...,Evidence is too weak for a grounded answer. Th...


## 7. Governance interpretation

This notebook is deliberately conservative. The RAG layer is useful because it makes the system explainable and searchable, but execution remains controlled by the original typed contracts, deterministic checks, distributional checks, and final shield status.